# 02 — Autoencoders with PyTorch

PyTorch-only companion to the original MNIST/Fashion-MNIST autoencoder notebook.

## 1. Dataset and normalization

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from pathlib import Path
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tf=transforms.ToTensor(); train=datasets.MNIST('data',train=True,download=True,transform=tf); test=datasets.MNIST('data',train=False,download=True,transform=tf)
loader=DataLoader(train,batch_size=256,shuffle=True)

## 2. Dense autoencoder

In [ ]:
class DenseAE(nn.Module):
 def __init__(self,latent=32):
  super().__init__(); self.encoder=nn.Sequential(nn.Flatten(),nn.Linear(784,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,latent)); self.decoder=nn.Sequential(nn.Linear(latent,64),nn.ReLU(),nn.Linear(64,128),nn.ReLU(),nn.Linear(128,784),nn.Sigmoid())
 def forward(self,x): return self.decoder(self.encoder(x)).view(-1,1,28,28)
model=DenseAE().to(device); opt=torch.optim.Adam(model.parameters()); loss_fn=nn.BCELoss()
for epoch in range(5):
 model.train(); total=0
 for x,_ in loader:
  x=x.to(device); opt.zero_grad(); recon=model(x); loss=loss_fn(recon,x); loss.backward(); opt.step(); total+=loss.item()*len(x)
 print(epoch+1,total/len(train))

## 3. Convolutional autoencoder

In [ ]:
class ConvAE(nn.Module):
 def __init__(self):
  super().__init__(); self.enc=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2)); self.dec=nn.Sequential(nn.Conv2d(8,8,3,padding=1),nn.ReLU(),nn.Upsample(scale_factor=2,mode='nearest'),nn.Conv2d(8,16,3,padding=1),nn.ReLU(),nn.Upsample(scale_factor=2,mode='nearest'),nn.Conv2d(16,1,3,padding=1),nn.Sigmoid())
 def forward(self,x): return self.dec(self.enc(x))
cae=ConvAE().to(device); opt=torch.optim.Adam(cae.parameters());
for epoch in range(5):
 for x,_ in loader:
  x=x.to(device); opt.zero_grad(); loss=loss_fn(cae(x),x); loss.backward(); opt.step()
 print('epoch',epoch+1,'last batch loss',float(loss))

## 4. Visualize reconstruction

In [ ]:
sample,_=test[0]; cae.eval()
with torch.inference_mode(): out=cae(sample.unsqueeze(0).to(device)).cpu().squeeze().numpy()
plt.figure(figsize=(6,3)); plt.subplot(1,2,1); plt.imshow(sample.squeeze(),cmap='gray'); plt.title('Original'); plt.axis('off'); plt.subplot(1,2,2); plt.imshow(out,cmap='gray'); plt.title('Reconstruction'); plt.axis('off')

## 5. Compression formulas and tips

Raw grayscale bits: \(H\\times W\\times b\). Quantized latent bits: \(N_z\\times q\). Approximate ratio must include metadata and model storage. MSE alone does not capture perceptual quality; add SSIM/PSNR. For categorical masks, resize with nearest-neighbour.